In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/twincheck'
OUT = f'{DRIVE}/F5_rerun/out_corrected_split'
CKPT = f'{DRIVE}/models/F5_corrected_split'
ROOT = '/content/genimage_root'
REPO = '/content/twincheck'
!rm -rf {REPO}
!git clone -q https://github.com/himaallu/Group-Project.git {REPO}
%cd {REPO}
!pip install -q onnx onnxruntime
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import pandas as pd
!mkdir -p {ROOT}/data/processed/vision
!ln -sfn {DRIVE}/data/processed/vision/genimage {ROOT}/data/processed/vision/genimage
for split in ('train', 'validation', 'test'):
    rows = pd.read_csv(f'data/processed/vision/genimage_{split}_manifest.csv')
    missing = [p for p in rows['exported_path'] if not os.path.exists(f'{ROOT}/{p}')]
    print(split, len(rows), rows.groupby('generator').size().to_dict(), 'missing', len(missing))
    assert not missing

In [ ]:
!python -m pytest -q tests/vision/test_f5.py
!python -m src.vision.train_detector --arch resnet50 --root {ROOT} --limit 8 --epochs 1 --batch 8 --ckpt-dir /content/check --out /content/check --no-wandb

In [ ]:
!mkdir -p {OUT}/results/F5 {OUT}/reports/figures
!rm -rf results/F5
!ln -sfn {OUT}/results/F5 results/F5

In [ ]:
for arch in ['resnet50', 'vit_b16']:
    !python -m src.vision.train_detector --arch {arch} --seed 42 --root {ROOT} --ckpt-dir {CKPT} --resume --out results/F5 --no-wandb

In [ ]:
!python -m src.vision.evaluate_detectors --root {ROOT} --ckpt-dir {CKPT} --skip-clip

In [ ]:
!python -m src.vision.gradcam --root {ROOT} --ckpt-dir {CKPT}
!cp reports/figures/fig_gradcam_resnet50.png reports/figures/fig_gradcam_vit_b16.png {OUT}/reports/figures/

In [ ]:
!python -m src.vision.export_onnx --ckpt-dir {CKPT}

In [ ]:
print(pd.read_csv('results/F5/cnn_vit.csv').to_string())
print(pd.read_csv('results/F5/onnx_latency.csv').to_string())
!ls -la {OUT}/results/F5 {OUT}/reports/figures {CKPT}